# CharWriteViT on Kaggle
OCR, Writer ID, Discriminator, and Character Classifier are trained jointly with the generator. Only `ngkinwang/iam-dataset` is required; character labels come directly from query-token-aligned decoder cells.

In [ ]:
REPOSITORY = 'https://github.com/YOUR_USER/YOUR_REPO.git'
!git clone $REPOSITORY /kaggle/working/CharWriteViT
%cd /kaggle/working/CharWriteViT
!pip install -q -e .

In [ ]:
# Inspect attached IAM files. EMNIST/fonts are not required for the joint run.
!find /kaggle/input -maxdepth 4 -type f | head -80

In [ ]:
# Shared helpers. The character classifier is not pretrained; train_iam updates it from query-labelled cells.
from pathlib import Path
import subprocess, sys
from IPython.display import display, Image

PROJECT = Path('/kaggle/working/CharWriteViT')
print('No classifier checkpoint or IAM character boxes are required.')

In [ ]:
# Optional only: train_classifier remains available for ablations/initialization,
# but the standard joint run below intentionally starts it from scratch.
CHARACTER_CLASSIFIER_PRETRAIN = None

In [ ]:
# Prepare IAM after attaching ngkinwang/iam-dataset.
subprocess.run([sys.executable, '-m', 'charwritevit.cli.prepare_iam',
                '--config', str(PROJECT / 'configs/kaggle.yaml')],
               cwd=PROJECT, check=True)
print((PROJECT / 'configs/kaggle.yaml').read_text())
print((Path('/kaggle/working/charwritevit-data/iam') / 'summary.json').read_text())

In [ ]:
# End-to-end IAM smoke test: D, OCR, Writer ID, Character Classifier, and G.
IAM_SMOKE = PROJECT / 'outputs' / 'iam-smoke'
subprocess.run([sys.executable, '-m', 'charwritevit.cli.train_iam',
                '--config', str(PROJECT / 'configs/kaggle.yaml'),
                '--output', str(IAM_SMOKE), '--epochs', '1', '--max-steps', '2'],
               cwd=PROJECT, check=True)
for name in ['sample_00_characters.png', 'sample_00_scales.png', 'sample_00_word.png']:
    display(Image(filename=str(IAM_SMOKE / 'preview' / 'epoch_000' / name)))

In [ ]:
# Full joint IAM training. If latest.pt was restored, all five networks resume together.
IAM_OUTPUT = PROJECT / 'outputs' / 'iam'
command = [sys.executable, '-m', 'charwritevit.cli.train_iam',
           '--config', str(PROJECT / 'configs/kaggle.yaml'),
           '--output', str(IAM_OUTPUT)]
latest = IAM_OUTPUT / 'latest.pt'
if latest.is_file(): command += ['--resume', str(latest)]
subprocess.run(command, cwd=PROJECT, check=True)

In [ ]:
# Evaluate the final checkpoint, including standard Inception FID and KID.
EVAL_OUTPUT = IAM_OUTPUT / 'evaluation'
subprocess.run([sys.executable, '-m', 'charwritevit.cli.evaluate',
                '--config', str(PROJECT / 'configs/kaggle.yaml'),
                '--checkpoint', str(IAM_OUTPUT / 'latest.pt'),
                '--output', str(EVAL_OUTPUT),
                '--max-batches', '100', '--kid-subset-size', '100'],
               cwd=PROJECT, check=True)
print((EVAL_OUTPUT / 'metrics.json').read_text())

In [ ]:
# Display loss curve and all 10 samples from the newest completed epoch.
display(Image(filename=str(IAM_OUTPUT / 'iam_curves.png')))
preview_epochs = sorted((IAM_OUTPUT / 'preview').glob('epoch_*'))
latest_preview = preview_epochs[-1]
for sample in range(10):
    for suffix in ('characters', 'scales', 'word'):
        artifact = latest_preview / f'sample_{sample:02d}_{suffix}.png'
        if artifact.is_file(): display(Image(filename=str(artifact)))

In [ ]:
# Archive checkpoints so the notebook Output can be saved as a Kaggle Dataset.
!tar -czf /kaggle/working/charwritevit-checkpoints.tgz outputs configs/kaggle.yaml